In [ ]:
from BASIS.modules import vis, imutils
import numpy as np
import matplotlib.pyplot as plt
from BASIS.models import base
import ehtim as eh
from BASIS.modules import likelihood


In [ ]:
# xsring/xsringauss parameters

fov = 300 # Field of view (uas)
imgdim = 128 # Image dimension (pixels)

xpoint = base.BaseModel(model_list=['point'], fov=fov, dim=imgdim)
xdisk = base.BaseModel(model_list=['disk'], fov=fov, dim=imgdim)
sdisk = base.BaseModel(model_list=['sdisk'], fov=fov, dim=imgdim)
xgauss = base.BaseModel(model_list=['gauss'], fov=fov, dim=imgdim)
xsr = base.BaseModel(model_list=['xsring'], fov=fov, dim=imgdim)
xsrg = base.BaseModel(model_list=['xsringauss'], fov=fov, dim=imgdim)
mrmodel = base.BaseModel(model_list=['mring4'], fov=fov, dim=imgdim)
combo = base.BaseModel(model_list=['xsringauss', 'gauss', 'gauss'], fov=fov, dim=imgdim, 
                       randomise_params=[False, False, True], random_seed=42)
grid = base.BaseModel(model_list=['pixelgrid'], fov=fov, dim=imgdim, randomise_params=[False])

models = {'point': xpoint, 'disk': xdisk, 'sdisk': sdisk, 'gauss': xgauss, 'xsring': xsr, 'xsringauss': xsrg, 'mring': mrmodel, 'combo': combo, 'pixelgrid': grid}

fig, axs = plt.subplots(1,len(models),figsize=(4*len(models),4))
for model in models:
    img =  models[model].sky_map()
    axs[list(models.keys()).index(model)].imshow(img, origin='lower', cmap='inferno')
    axs[list(models.keys()).index(model)].set_title(model)
plt.tight_layout()


In [ ]:
# Compare analytical and numerical visibilities
model_choice = grid

eh_img = eh.image.make_empty(imgdim, fov=fov*1e-6/206265, ra=0, dec=0)
img = np.roll(model_choice.sky_map()*1, 0)
# img = np.ones_like(img)
# img /= np.sum(img)
eh_img._imdict['I'] = img
fig, ax= plt.subplots()
im = ax.imshow(np.flipud(eh_img.imarr()), cmap='afmhot')
plt.colorbar(im, ax=ax, label='Jy/pixel')



# obs = eh.obsdata.load_uvfits('../data/uvfits/SR1_M87_2017_101_hilo_hops_netcal_StokesI.uvfits')
obs = eh.obsdata.load_uvfits('../data/uvfits/test_M87_2017_101_225_composite.uvfits')
eh_img.ra = obs.ra
eh_img.dec = obs.dec
eh_img.rf = obs.rf
eh_img.psize = eh_img.psize
obs = eh_img.observe_same_nonoise(obs, ttype='fast')

uv = np.array([obs.data['u'], obs.data['v']])

numVis = vis.DFT(eh_img.imarr().reshape(1, imgdim, imgdim), np.transpose(uv), xfov=fov, yfov=fov)
anaVis = model_choice.sample_vis(uv, ttype='analytical')
obs.plotall('u','v', conj=True) # uv coverage

fig, axs = plt.subplots(1,2,figsize=(10,4))
uvdist = np.sqrt(uv[0]**2 + uv[1]**2)
axs[0].scatter(uvdist, np.abs(obs.data['vis']), color='blue', s=5, label='EHT')
axs[0].scatter(uvdist, np.abs(anaVis), color='red', s=5, label='Analytic')
axs[0].scatter(uvdist, np.abs(numVis), marker='x', color='green', s=1, label='DFT')

axs[1].scatter(uvdist, np.angle(obs.data['vis']), color='blue', s=5, label='EHT')
axs[1].scatter(uvdist, np.angle(anaVis), color='red', s=5, label='Analytic')
axs[1].scatter(uvdist, np.angle(numVis), marker='x', color='green',  s=2, label='DFT')

axs[0].legend()

# obs.save_uvfits('../data/uvfits/test_mring.uvfits')




In [ ]:
# Custom likelihood, check all match
noise_factor = 1
static_noise_floor = 0.0001
noise_frac = 0.05
custom_likelihood = likelihood.ModelLikelihood(model_names=model_choice.model_list, obs=obs, count='min', imgdim=imgdim, fov=fov,
                                               noise_factor=noise_factor, static_noise_floor=static_noise_floor, noise_frac=noise_frac)

custom_likelihood.plot_all(model_choice.params)

# test = model_choice.params.copy()
# custom_likelihood.dterms = {'camp':0.1}
# print(custom_likelihood.log_likelihood(test))

In [ ]:
# generate, read the parameter file

# imgdim = 128
# fov = 100000

# # model_components = ['point', 'point', 'point', 'point', 'point']
# model_components = ['gauss', 'gauss', 'gauss']

# base_model = base.BaseModel(model_list=model_components, fov=fov, dim=imgdim)
# # # base_model.save_params_to_file('params/GRAVITY_ppppp.json')
# base_model.load_params_from_file('params/TANAMI_ggg.json')

# fig, ax = plt.subplots()
# ax.imshow(base_model.sky_map(), origin='lower', cmap='inferno')